In [2]:
from tensorflow.keras.layers import SimpleRNN,LSTM,GRU,Bidirectional,Dense,Embedding
from tensorflow.keras.models import Sequential
from tensorflow.keras.datasets import imdb
import numpy as np

In [3]:
#getting reviews with words that come under 5000
# most occuring words in the entire
# corpus of textul review data
vocab_size=50000
(x_train,y_train),(x_test,y_test)=imdb.load_data(num_words=vocab_size)
print(x_train[0])

[1, 14, 22, 16, 43, 530, 973, 1622, 1385, 65, 458, 4468, 66, 3941, 4, 173, 36, 256, 5, 25, 100, 43, 838, 112, 50, 670, 22665, 9, 35, 480, 284, 5, 150, 4, 172, 112, 167, 21631, 336, 385, 39, 4, 172, 4536, 1111, 17, 546, 38, 13, 447, 4, 192, 50, 16, 6, 147, 2025, 19, 14, 22, 4, 1920, 4613, 469, 4, 22, 71, 87, 12, 16, 43, 530, 38, 76, 15, 13, 1247, 4, 22, 17, 515, 17, 12, 16, 626, 18, 19193, 5, 62, 386, 12, 8, 316, 8, 106, 5, 4, 2223, 5244, 16, 480, 66, 3785, 33, 4, 130, 12, 16, 38, 619, 5, 25, 124, 51, 36, 135, 48, 25, 1415, 33, 6, 22, 12, 215, 28, 77, 52, 5, 14, 407, 16, 82, 10311, 8, 4, 107, 117, 5952, 15, 256, 4, 31050, 7, 3766, 5, 723, 36, 71, 43, 530, 476, 26, 400, 317, 46, 7, 4, 12118, 1029, 13, 104, 88, 4, 381, 15, 297, 98, 32, 2071, 56, 26, 141, 6, 194, 7486, 18, 4, 226, 22, 21, 134, 476, 26, 480, 5, 144, 30, 5535, 18, 51, 36, 28, 224, 92, 25, 104, 4, 226, 65, 16, 38, 1334, 88, 12, 16, 283, 5, 16, 4472, 113, 103, 32, 15, 16, 5345, 19, 178, 32]


In [4]:
word_idx=imdb.get_word_index()
#orginally the index number of a value not a key,
# hence converting the index as key and the words as values
word_idx = {i:word for word,i in word_idx.items()} 
#again printing the review
print([word_idx[i] for i in x_train[0]])

['the', 'as', 'you', 'with', 'out', 'themselves', 'powerful', 'lets', 'loves', 'their', 'becomes', 'reaching', 'had', 'journalist', 'of', 'lot', 'from', 'anyone', 'to', 'have', 'after', 'out', 'atmosphere', 'never', 'more', 'room', 'titillate', 'it', 'so', 'heart', 'shows', 'to', 'years', 'of', 'every', 'never', 'going', 'villaronga', 'help', 'moments', 'or', 'of', 'every', 'chest', 'visual', 'movie', 'except', 'her', 'was', 'several', 'of', 'enough', 'more', 'with', 'is', 'now', 'current', 'film', 'as', 'you', 'of', 'mine', 'potentially', 'unfortunately', 'of', 'you', 'than', 'him', 'that', 'with', 'out', 'themselves', 'her', 'get', 'for', 'was', 'camp', 'of', 'you', 'movie', 'sometimes', 'movie', 'that', 'with', 'scary', 'but', 'pratfalls', 'to', 'story', 'wonderful', 'that', 'in', 'seeing', 'in', 'character', 'to', 'of', '70s', 'musicians', 'with', 'heart', 'had', 'shadows', 'they', 'of', 'here', 'that', 'with', 'her', 'serious', 'to', 'have', 'does', 'when', 'from', 'why', 'what', 

In [5]:
#get minimum and maximum length of reviews
print("max length of a review: ",len(max((x_train+x_test),key=len)))
print("min length of a review: ",len(min((x_train+x_test),key=len)))

max length of a review:  2697
min length of a review:  70


In [6]:
from tensorflow.keras.preprocessing import sequence
#keeping a fixed length of all reviews to max 400 words
max_words=400

x_train=sequence.pad_sequences(x_train,maxlen=max_words)
x_test=sequence.pad_sequences(x_test,maxlen=max_words)

x_valid,y_valid = x_train[:64],y_train[:64]
x_train,y_train = x_train[64:],y_train[64:]

In [7]:
#building simple RNN
#fixing every word's embedding size to b 32
embd_len = 32
#creating a RNN model
RNN_model = Sequential(name="Simple_RNN")
RNN_model.add(Embedding(vocab_size,embd_len,input_length=max_words))
#in case of a stacked(more than layer of RNN)
#use return_sequence=True
RNN_model.add(SimpleRNN(128,activation="tanh",return_sequences=False))
RNN_model.add(Dense(1,activation="sigmoid"))

#printing model summary
print(RNN_model.summary())

#compiling model
RNN_model.compile(loss='binary_crossentropy',optimizer='adam',metrics=['accuracy'])

# training the model
history=RNN_model.fit(x_train,y_train,batch_size=64,epochs=5,verbose=1,validation_data=(x_valid,y_valid))

#printing the model on test data
print()
print("Simple_RNN Score--->", RNN_model.evaluate(x_test,y_test,verbose=0))

Model: "Simple_RNN"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 embedding (Embedding)       (None, 400, 32)           1600000   
                                                                 
 simple_rnn (SimpleRNN)      (None, 128)               20608     
                                                                 
 dense (Dense)               (None, 1)                 129       
                                                                 
Total params: 1,620,737
Trainable params: 1,620,737
Non-trainable params: 0
_________________________________________________________________
None
Epoch 1/5
390/390 [==============================] - 21s 54ms/step - loss: 0.6899 - accuracy: 0.5346 - val_loss: 0.6565 - val_accuracy: 0.6094
Epoch 2/5
390/390 [==============================] - 21s 53ms/step - loss: 0.6209 - accuracy: 0.6569 - val_loss: 0.6147 - val_accuracy: 0.6562
Epoch 3/5
390/390 [

In [8]:
# Defining GRU Model
gru_model=Sequential(name="GRU_model")
gru_model.add(Embedding(vocab_size,embd_len,input_length=max_words))
gru_model.add(GRU(128,activation='tanh',return_sequences=False))
gru_model.add(Dense(1,activation='sigmoid'))

# printing model summary
print(gru_model.summary())

#Compiling the model
gru_model.compile(loss='binary_crossentropy',optimizer='adam',metrics=['accuracy'])

# training the model

# training the GRU model
history2=gru_model.fit(x_train,y_train,batch_size=64,epochs=5,verbose=1,validation_data=(x_valid,y_valid))
#printing model score on test data
print()
print("GRU_model Score--->", gru_model.evaluate(x_test,y_test,verbose=0))

Model: "GRU_model"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 embedding_1 (Embedding)     (None, 400, 32)           1600000   
                                                                 
 gru (GRU)                   (None, 128)               62208     
                                                                 
 dense_1 (Dense)             (None, 1)                 129       
                                                                 
Total params: 1,662,337
Trainable params: 1,662,337
Non-trainable params: 0
_________________________________________________________________
None
Epoch 1/5
390/390 [==============================] - 119s 303ms/step - loss: 0.4833 - accuracy: 0.7510 - val_loss: 0.3093 - val_accuracy: 0.8750
Epoch 2/5
390/390 [==============================] - 118s 303ms/step - loss: 0.2483 - accuracy: 0.9057 - val_loss: 0.2575 - val_accuracy: 0.9062
Epoch 3/5
390/39

In [9]:
#defining LSTM Model

lstm_model = Sequential(name="LSTM_Model")
lstm_model.add(Embedding(vocab_size,embd_len,input_length=max_words))
lstm_model.add(LSTM(128,activation='relu',return_sequences=False))

lstm_model.add(Dense(1,activation='sigmoid'))

# printing model summary
print(lstm_model.summary())

#Compiling the model
lstm_model.compile(loss='binary_crossentropy',optimizer='adam',metrics=['accuracy'])

# training the model

# training the lstm_model
history3=lstm_model.fit(x_train,y_train,batch_size=64,epochs=5,verbose=1,validation_data=(x_valid,y_valid))
#printing model score on test data
print()
print("lstm_model Score--->", lstm_model.evaluate(x_test,y_test,verbose=0))

Model: "LSTM_Model"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 embedding_2 (Embedding)     (None, 400, 32)           1600000   
                                                                 
 lstm (LSTM)                 (None, 128)               82432     
                                                                 
 dense_2 (Dense)             (None, 1)                 129       
                                                                 
Total params: 1,682,561
Trainable params: 1,682,561
Non-trainable params: 0
_________________________________________________________________
None
Epoch 1/5
390/390 [==============================] - 123s 313ms/step - loss: nan - accuracy: 0.5070 - val_loss: nan - val_accuracy: 0.6094
Epoch 2/5
390/390 [==============================] - 124s 317ms/step - loss: nan - accuracy: 0.4997 - val_loss: nan - val_accuracy: 0.6094
Epoch 3/5
390/390 [========